In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Works if the notebook is inside notebooks/
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "peft"

print(PROJECT_ROOT)

In [ ]:
LANGUAGES = {
    "de": "German",
    "sv": "Swedish",
}

CONDITIONS = {
    "english": "English-only",
    "target": "Target-only",
    "bilingual": "Bilingual",
}

RUN_NAME = "baseline_lora"

In [ ]:
all_results = {}

for language in LANGUAGES:
    all_results[language] = {}

    for condition in CONDITIONS:
        path = (
            OUTPUT_DIR
            / language
            / condition
            / RUN_NAME
            / "dev_metrics.json"
        )

        if path.exists():
            with path.open("r", encoding="utf-8") as file:
                all_results[language][condition] = json.load(file)

            print(f"Loaded: {language} / {condition}")
        else:
            print(f"Missing: {path}")

In [ ]:
main_metrics = [
    "micro_f1",
    "jaccard",
    "macro_f1",
    "precision",
    "recall",
    "hamming_loss",
]

rows = []

for language, language_name in LANGUAGES.items():
    for condition, condition_name in CONDITIONS.items():

        result = all_results[language].get(condition)

        if result is None:
            continue

        # We always want performance on the target language here.
        if language not in result:
            continue

        metrics = result[language]

        row = {
            "Language": language_name,
            "Condition": condition_name,
        }

        for metric in main_metrics:
            row[metric] = metrics[metric]

        rows.append(row)

target_df = pd.DataFrame(rows)

target_df

In [ ]:
target_df.round(3)

In [ ]:
micro_df = target_df.pivot(
    index="Condition",
    columns="Language",
    values="micro_f1",
)

micro_df.plot(
    kind="bar",
    figsize=(8, 5),
)

plt.ylabel("Micro F1")
plt.xlabel("Training condition")
plt.title("Target-language PEFT performance")
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.grid(axis="y", alpha=0.3)
plt.legend(title="Evaluation language")
plt.show()

In [ ]:
jaccard_df = target_df.pivot(
    index="Condition",
    columns="Language",
    values="jaccard",
)

jaccard_df.plot(
    kind="bar",
    figsize=(8, 5),
)

plt.ylabel("Jaccard Score")
plt.xlabel("Training condition")
plt.title("Target-language Jaccard performance")
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.grid(axis="y", alpha=0.3)
plt.legend(title="Evaluation language")
plt.show()

In [ ]:
english_rows = []

for language, language_name in LANGUAGES.items():
    for condition in ["english", "bilingual"]:

        result = all_results[language].get(condition)

        if result is None or "en" not in result:
            continue

        metrics = result["en"]

        row = {
            "Target experiment": language_name,
            "Condition": CONDITIONS[condition],
        }

        for metric in main_metrics:
            row[metric] = metrics[metric]

        english_rows.append(row)

english_df = pd.DataFrame(english_rows)

english_df.round(3)

In [ ]:
transfer_rows = []

for language, language_name in LANGUAGES.items():

    result = all_results[language].get("english")

    if result is None:
        continue

    if "en" not in result or language not in result:
        continue

    en_metrics = result["en"]
    target_metrics = result[language]

    transfer_rows.append(
        {
            "Target language": language_name,
            "English Micro F1": en_metrics["micro_f1"],
            "Target Micro F1": target_metrics["micro_f1"],
            "Micro F1 drop": (
                en_metrics["micro_f1"]
                - target_metrics["micro_f1"]
            ),
            "English Jaccard": en_metrics["jaccard"],
            "Target Jaccard": target_metrics["jaccard"],
            "Jaccard drop": (
                en_metrics["jaccard"]
                - target_metrics["jaccard"]
            ),
        }
    )

transfer_df = pd.DataFrame(transfer_rows)

transfer_df.round(3)

In [ ]:
emotion_rows = []

for language, language_name in LANGUAGES.items():
    for condition, condition_name in CONDITIONS.items():

        result = all_results[language].get(condition)

        if result is None or language not in result:
            continue

        emotion_scores = result[language]["per_emotion_f1"]

        for emotion, score in emotion_scores.items():
            emotion_rows.append(
                {
                    "Language": language_name,
                    "Condition": condition_name,
                    "Emotion": emotion,
                    "F1": score,
                }
            )

emotion_df = pd.DataFrame(emotion_rows)

emotion_df.head()

In [ ]:
german_emotions = (
    emotion_df[
        emotion_df["Language"] == "German"
    ]
    .pivot(
        index="Emotion",
        columns="Condition",
        values="F1",
    )
)

german_emotions.round(3)

german_emotions.plot(
    kind="bar",
    figsize=(11, 5),
)

plt.ylabel("F1")
plt.xlabel("Emotion")
plt.title("German per-emotion F1")
plt.xticks(rotation=45)
plt.ylim(0, 1)
plt.grid(axis="y", alpha=0.3)
plt.legend(title="Training condition")
plt.show()

In [ ]:
swedish_emotions = (
    emotion_df[
        emotion_df["Language"] == "Swedish"
    ]
    .pivot(
        index="Emotion",
        columns="Condition",
        values="F1",
    )
)

swedish_emotions.round(3)

swedish_emotions.plot(
    kind="bar",
    figsize=(11, 5),
)

plt.ylabel("F1")
plt.xlabel("Emotion")
plt.title("Swedish per-emotion F1")
plt.xticks(rotation=45)
plt.ylim(0, 1)
plt.grid(axis="y", alpha=0.3)
plt.legend(title="Training condition")
plt.show()

In [ ]:
summary = target_df[
    [
        "Language",
        "Condition",
        "micro_f1",
        "jaccard",
        "macro_f1",
        "precision",
        "recall",
        "hamming_loss",
    ]
].copy()

summary = summary.sort_values(
    ["Language", "Condition"]
)

summary.round(3)